# 21 — KG-Native Analytics

KG-native analytics over the configured Fuseki dataset:

1. **POI centrality & structural roles** (co-visitation + flow projections), materialized back to the KG (`trajecto:pageRank`, `trajecto:betweennessCentrality`, `trajecto:communityId`, `trajecto:hasCentralityRole`).
2. **Multi-hop relatedness** — path-length distribution and explicit evidence chains between POIs.
3. **Class-level aggregation** — subsumption-aware contingency tables with chi-square tests.
4. **Temporal-sliced centrality** — per-context PageRank shift and community stability (NMI).

Logic lives in `helpers/kg_analytics.py`; this notebook orchestrates and reports. Outputs land in `output/analytics/`.

In [ ]:
import os
import sys
import json
from pathlib import Path
sys.path.insert(0, '.')

from helpers import kg_analytics as ka

# TRAJECTO_ANALYTICS_DIR scopes analytics outputs per corpus, so a second
# instantiation does not overwrite the first corpus's centrality tables.
OUT = Path(os.environ.get('TRAJECTO_ANALYTICS_DIR', './output/analytics'))
OUT.mkdir(parents=True, exist_ok=True)
records = ka.load_interaction_records()
meta = ka.load_poi_metadata()
contexts = ka.load_temporal_contexts()
print(f'{records.traj.nunique()} trajectories, {records.poi.nunique()} POIs, '
      f'{len(records)} meaningful interactions, {len(contexts)} trajectories with temporal context')

## 1 · Centrality, communities, structural roles

In [ ]:
covisit, flow = ka.build_graphs(records)
print(f'covisit: {covisit.number_of_nodes()} nodes / {covisit.number_of_edges()} edges | '
      f'flow: {flow.number_of_nodes()} nodes / {flow.number_of_edges()} edges')
cent = ka.centrality_analysis(covisit, flow, meta)
cent.to_csv(OUT / 'poi_centrality.csv', index=False)
print(cent['role'].value_counts().to_string())
cent.head(15)[['label','category','covisit_degree','betweenness','pagerank','flow_in','flow_out','community','role']]

In [ ]:
written = ka.materialize_centrality(cent)
print(f'{written} centrality triples materialized to the KG '
      f'(cleanup-safe via MATERIALIZED_PREDICATES)')

## 2 · Multi-hop relatedness

In [ ]:
dist = ka.path_length_distribution(covisit, flow)
print('POI-pair shortest-path length distribution (giant component, cutoff 6):')
print((dist / dist.sum()).round(3).to_string())
top = cent.head(6)['poi'].tolist()
for a, b in [(top[0], top[3]), (top[1], top[4]), (top[2], top[5])]:
    chain = ka.explain_relation(a, b, covisit, flow, meta)
    if chain:
        print('\n' + ' | '.join(chain))

## 3 · Class-level aggregation (chi-square)

In [ ]:
tables = ka.class_level_tables()
for name, res in tables.items():
    print(f'=== {name}  chi2={res["chi2"] if res["chi2"] is None else round(res["chi2"],1)}  p={res["p"] if res["p"] is None else format(res["p"],".2e")} ===')
    if not res['table'].empty:
        print(res['table'].to_string())
        res['table'].to_csv(OUT / f'class_{name}.csv')
    print()

## 4 · Temporal-sliced centrality

In [ ]:
shift, nmi = ka.temporal_centrality(records, contexts, meta)
shift.to_csv(OUT / 'temporal_centrality.csv', index=False)
slice_cols = [c for c in shift.columns if c in ka.TEMPORAL_SLICES]
print('Top POIs by mean PageRank across temporal slices:')
print(shift.head(10)[['label','category'] + slice_cols + ['cv']].to_string(index=False))
print('\nMost temporally volatile hubs (high cv among top-50 by mean PageRank):')
print(shift.head(50).sort_values('cv', ascending=False).head(8)[['label','category','cv']].to_string(index=False))
print('\nCommunity stability across slices (NMI on shared POIs):')
print(json.dumps(nmi, indent=1))

## Summary

- Structural roles are materialized in the KG: downstream SPARQL (and the recommender's explanation layer) can reference `trajecto:hasCentralityRole` directly — a candidate future signal for cold-start scoring.
- The path-length distribution quantifies how many hops separate POI pairs; short chains are the interpretable explanations the design document calls for.
- The chi-square tables surface which schema-level associations are statistically real (e.g. purpose × interaction type) rather than anecdotal.
- Temporal centrality shift identifies time-specific attractors (high coefficient of variation) vs. stable hubs; NMI quantifies whether the community structure persists across time contexts.